<a href="https://colab.research.google.com/github/frasercrichton/ai-dde-hackthon/blob/feature%2Fleiden-guidelines-doc/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install  chromadb
!pip install git+https://github.com/huggingface/transformers.git triton

import logging


# Remove existing handlers (prevents duplicate logs)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Set up logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.1 MB/s eta 0:00:0

In [17]:
from transformers import AutoTokenizer, AutoModel
from langchain.text_splitter import RecursiveCharacterTextSplitter
import torch

class EmbeddingsProcessor:

    def __init__(self, model_name):

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()


    def create_embeddings(self, text):

        inputs = self.tokenizer(
            text,
            return_tensors='pt',
            truncation=True
        )
        logger.info(f'inputs: {inputs}')

        if torch.cuda.is_available():
            logger.info('cuda available')
            self.model.to('cuda')
            inputs = {k: v.to('cuda') for k, v in inputs.items()}
        else:
            logger.warning('cuda not available!')


        with torch.no_grad():
            outputs = self.model(**inputs)

        return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy().tolist()


In [13]:
import chromadb


class RAGDatabase:

    def __init__(self, collection_name):
        self.client = chromadb.Client()
        self.collection = self.client.get_or_create_collection(name=collection_name)

    def store_documents(self, documents: list):

        kwargs = {
            "documents": [doc.get('text') for doc in documents],
            "embeddings": [doc.get('embedding') for doc in documents],
            "ids": [doc.get('id') for doc in documents]
        }

        metadata = [doc.get('metadata', {}) for doc in documents if doc.get('metadata') is not None]

        if len(metadata) > 0:
            kwargs["metadatas"] = metadata

        try:
            self.collection.add(**kwargs)
        except Exception as e:
            logging.error(f"Error adding documents: {e}")


    def find_relevant_documents(
        self,
        query: str,
        filters: dict,
        n_results=3,
    ):

        # print(
        #     f'find_relevant_documents query: {query} headers: {headers}, subheaders {subheaders}, context: {context}'
        # )

        return self.collection_query(query, filters, n_results)

    def query_with_embeddings(self, query_embedding, filter_dict=None, n_results=3):

# n_results=3 to 5 is generally a safe default for most applications.
# What If My Top n_results Are Not Useful?
# 	•	Increase the embedding quality → Use a better embedding model or fine-tune one.
# 	•	Apply re-ranking → Use a second model to score and reorder the results.
# 	•	Filter results by metadata → If using ChromaDB with metadata, filter based on relevant categories (e.g., category="landmarks").
# 	•	Use hybrid retrieval → Combine keyword-based search with embeddings for better results.

        # n_results=3
        # high precision 1-3
  #       Works well when information may be spread across multiple short documents.
	# •	Example: Answering questions that require synthesizing different perspectives, like a summary of multiple research papers.

        # more context 3 to 5
  #       # high recall (broad retrieval for re-ranking) 5 to 10+
  #       Recommended when re-ranking or filtering is applied after retrieval.
	# •	Example: Open-domain Q&A systems where an LLM will decide the most relevant information after fetching multiple candidates.

        query = {
            'query_embeddings': query_embedding,
            'n_results': n_results
        }

        if filter_dict:
            query['where'] = filter_dict

        results = self.collection.query(**query)

        return [
            {
                'text': doc_text,
                'id': doc_id,
                'metadata': doc_metadata
            }
            for doc_text, doc_id, doc_metadata in zip(results['documents'][0], results['ids'][0], results['metadatas'][0])
        ]


    def collection_query(self, query, filter, n_results=3):

        results = self.collection.query(
            query_texts=[query], where=filter, n_results=n_results
        )
        # print(f'collection_query : {results}')
        return [
            {
                'text': doc_text,
                'id': results['ids'][0][i],
                'metadata': results['metadatas'][0][i],
            }
            for i, doc_text in enumerate(results['documents'][0])
        ]

    def delete_collection(self, collection_name):
        self.client.delete_collection(collection_name)

In [2]:
class LeidenGuidelinesMetadata:

    def createFilters(self, headers, subheaders, context):

      filter_dict = {}

      if headers:
          filter_dict['header'] = {'$in': headers}

      if subheaders:
          filter_dict['subheader'] = {'$in': subheaders}

      if context:
          filter_dict['context'] = {'$in': context}

      if filter_dict:
          return {'$or': [{key: value} for key, value in filter_dict.items()]}
      else:
          return None




In [5]:
chunk = """
Keywords: relevance; probative value; prejudice; reliability; chain of custody Prima facie authenticity must be demonstrated before audio recordings can be admitted into evidence. The ICC Trial Chamber in Bemba noted that ‘unless the [Radio France Internationale (RFI) audio recording] bears sufficient indicia that it is what it purports to be (in this case, an RFI transmission), the prosecution must also provide information on its source, originality and integrity’.283 Since this information was absent, the probative value of the recording ‘was outweighed by its potentially prejudicial effect on a fair trial’ and its admission was rejected.284 280 Prosecutor v Bemba (Public Redacted Version of “Decision on the Prosecution’s Application for Admission of Materials into Evidence Pursuant to Article 64(9) of the Rome Statute” of 6 September 2012) ICC-01/05-01/08-2299-Red (8 October 2012) (TC III) [101]. 281 Prosecutor v Bemba (Public Redacted Version of “Decision on the Prosecution’s Application for Admission of Materials into Evidence Pursuant to Article 64(9) of the Rome Statute” of 6 September 2012) ICC-01/05-01/08-2299-Red (8 October 2012) (TC III) [123]-[124]. 282 Prosecutor v Bemba (Public Redacted Version of “Decision on the Prosecution’s Application for Admission of Materials into Evidence Pursuant to Article 64(9) of the Rome Statute” of 6 September 2012) ICC-01/05-01/08-2299-Red (8 October 2012) (TC III) [125]-[126]. 283 Prosecutor v Bemba (Public Redacted Version of “Decision on the Prosecution’s Application for Admission of Materials into Evidence Pursuant to Article 64(9) of the Rome Statute” of 6 September 2012) ICC-01/05-01/08-2299-Red (8 October 2012) (TC III) [122]. 284 Prosecutor v Bemba (Public Redacted Version of “Decision on the Prosecution’s Application for Admission of Materials into Evidence Pursuant to Article 64(9) of the Rome Statute” of 6 September 2012) ICC-01/05-01/08-2299-Red (8 October 2012) (TC III) [122]. 52 Open Source Audio Recordings of Media Broadcasts. The ICC Trial Chamber in Bemba held that where the audio recording of an interview lacks a date and contains no questions, the tendering party must provide sufficient information to identify the recorded voice and ‘to confirm the date, circumstances and context in which the recording was created’.285 In the absence of such information, the ICC Trial Chamber found that it could not afford probative value to audio recording CAR-DEF-0001-0830, which the Prosecution alleged was a monologue of the Secretary-General of the Movement for the Liberation of the Congo.286 Moreover, in the context of the ICTY proceedings against Mladić, and pursuant to Rule 89(D) of the ICTY Rules of Procedure and Evidence,287 ‘[the] Chamber may exclude evidence if its probative value is substantially outweighed by the need to ensure a fair trial’. Nevertheless, open source audio recordings may be admitted if counsel show with sufficient clarity and specificity the relevance and probative value of these documents, and how they fit into the case.288 In Mladić, the ICTY Prosecution requested the admission of open source local and international radio news reports from the bar table.289 The Defence objected to their admission on the grounds that they originated from an open source and as such the author was unknown, rendering the Defence unable to challenge the content of the material, and that it was unclear whether the source heard the information from others.290 The ICTY Trial Chamber found that the general Defence submissions in relation to the origin of these documents were insufficient to successfully challenge their probative value, or preclude admission pursuant to Rule 89(D) of the Rules. Having considered the documents in this category, the Chamber was satisfied that the Prosecution had shown with sufficient clarity and specificity the relevance and probative value of each of these documents, and how they fit into its case.291 285 Prosecutor v Bemba (Public Redacted Version of “Decision on the Prosecution’s Application for Admission of Materials into Evidence Pursuant to Article 64(9) of the Rome Statute” of 6 September 2012) ICC-01/05-01/08-2299-Red (8 October 2012) (TC III) [84]. 286 Prosecutor v Bemba (Public Redacted Version of “Decision on the Prosecution’s Application for Admission of Materials into Evidence Pursuant to Article 64(9) of the Rome Statute” of 6 September 2012) ICC-01/05-01/08-2299-Red (8 October 2012) (TC III) [82], [84]. 287 cf Article 69(4) of the Rome Statute. 288 Prosecutor v Mladić (Decision on Prosecution Motion for Admission of Documents from the Bar Table (Municipalities Component)) IT-09-92 (11 February 2014) (TC) [9]. 289 Prosecutor v Mladić (Decision on Prosecution Motion for Admission of Documents from the Bar Table (Municipalities Component)) IT-09-92 (11 February 2014) (TC) [1]. 290 Prosecutor v Mladić (Decision on Prosecution Motion for Admission of Documents from the Bar Table (Municipalities Component)) IT-09-92 (11 February 2014) (TC) [7]. 291 Prosecutor v Mladić (Decision on Prosecution Motion for Admission of Documents from the Bar Table (Municipalities Component)) IT-09-92 (11 February 2014) (TC) [8]. 53'
"""


In [18]:
rag_database = RAGDatabase(collection_name='my_collection')

embeddings_processor = EmbeddingsProcessor('sentence-transformers/all-MiniLM-L6-v2')

embedding = embeddings_processor.create_embeddings(chunk)

rag_database.store_document(doc_id='test-x', document= chunk, embedding= embedding)


INFO: inputs: {'input_ids': tensor([[  101,  3145, 22104,  1024, 21923,  1025,  4013, 14479,  3512,  3643,
          1025, 18024,  1025, 15258,  1025,  4677,  1997,  9968, 21111,  6904,
         23402, 21452,  2442,  2022,  7645,  2077,  5746,  5633,  2064,  2022,
          4914,  2046,  3350,  1012,  1996, 16461,  3979,  4574,  1999,  2022,
         11201,  3264,  2008,  1520,  4983,  1996,  1031,  2557,  2605, 21339,
          1006, 21792,  2072,  1007,  5746,  3405,  1033,  6468,  7182, 27427,
         24108,  2008,  2009,  2003,  2054,  2009, 16405, 14536, 11589,  2015,
          2000,  2022,  1006,  1999,  2023,  2553,  1010,  2019, 21792,  2072,
          6726,  1007,  1010,  1996, 11537,  2442,  2036,  3073,  2592,  2006,
          2049,  3120,  1010,  2434,  3012,  1998, 11109,  1521,  1012, 25504,
          2144,  2023,  2592,  2001,  9962,  1010,  1996,  4013, 14479,  3512,
          3643,  1997,  1996,  3405,  1520,  2001,  2041, 27204,  9072,  2011,
          2049,  9280,  

AttributeError: 'RAGDatabase' object has no attribute 'store_document'

In [14]:
rag_database = RAGDatabase(collection_name='my_collection')
# rag_database.delete_all(collection_name='my_collection')
documents =[
    {'text': 'The Eiffel Tower is located in Paris, France.'},
    {'text': 'The Great Wall of China is one of the Seven Wonders of the World.'},
    {'text': 'Python is a popular programming language for data science.'},
    {'text': 'Leonardo da Vinci painted the Mona Lisa.'},
]

for i, doc in enumerate(documents):
    doc['id'] = i


documents = [
    {
        **document,
        'id': str(i),
        'embedding': embeddings_processor.create_embeddings(document['text'])
    }
    for i, document in enumerate(documents)
]

print(documents)

rag_database.store_documents(documents)
query_embeddings = embeddings_processor.create_embeddings("who painted the Mona Lisa?")
x = rag_database.query_with_embeddings(query_embeddings)
logger.info(f'search result {x}')

INFO: inputs: {'input_ids': tensor([[  101,  1996,  1041, 13355,  2884,  3578,  2003,  2284,  1999,  3000,
          1010,  2605,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
INFO: inputs: {'input_ids': tensor([[  101,  1996,  2307,  2813,  1997,  2859,  2003,  2028,  1997,  1996,
          2698, 16278,  1997,  1996,  2088,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
INFO: inputs: {'input_ids': tensor([[  101, 18750,  2003,  1037,  2759,  4730,  2653,  2005,  2951,  2671,
          1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
INFO: inputs: {'input_ids': tensor([[  101, 14720,  4830, 23765,  4993,  1996, 13813,  7059,  1012,   102]]), 'tok

[{'text': 'The Eiffel Tower is located in Paris, France.', 'id': '0', 'embedding': [0.25255128741264343, 0.261593759059906, 0.04478899762034416, -0.004811152815818787, 0.1809592992067337, -0.06470618396997452, -0.41695189476013184, 0.044289134442806244, 0.20888380706310272, -0.09429414570331573, 0.11149588972330093, -0.18263275921344757, 0.3690599501132965, -0.48395201563835144, 0.03566158190369606, -0.2918674051761627, -0.07082509249448776, -0.12295816838741302, 0.06394852697849274, -0.3695654571056366, 0.3303406834602356, -0.5041595101356506, 0.21871860325336456, -0.08313480764627457, -0.3599214255809784, -0.09521540254354477, -0.33874282240867615, 0.13332165777683258, -0.07765697687864304, -0.26415830850601196, 0.2931792438030243, -0.16103528439998627, -0.3515162765979767, 0.2917000353336334, -0.16635742783546448, 0.14924100041389465, 0.15074694156646729, -0.37358352541923523, 0.010461081750690937, -0.13055160641670227, 0.03111202083528042, 0.061918314546346664, -0.08030898123979568

In [3]:
x = [
    {'headers': 'VIDEOS', 'subheaders': '', 'context' :'' },
    {'headers': 'VIDEOS', 'subheaders': '', 'context' :'' },
    {'headers': 'VIDEOS', 'subheaders': '', 'context' :'' },
    {'headers': 'PHOTOGRAPHS', 'subheaders': '', 'context' :'' },
    {'headers': 'PHOTOGRAPHS', 'subheaders': '', 'context' :'' },
    {'headers': 'PHOTOGRAPHS', 'subheaders': '', 'context' :'' },
    {'headers': 'AERIAL AND SATELLITE IMAGES', 'subheaders': '', 'context' :'' },
    {'headers': 'AERIAL AND SATELLITE IMAGES', 'subheaders': '', 'context' :'' },
    {'headers': 'AERIAL AND SATELLITE IMAGES', 'subheaders': '', 'context' :'' },
    ]


x = LeidenGuidelinesMetadata().createFilters(headers='header text', subheaders='subheaders text', context='context text')

print(x)

{'$or': [{'header': {'$in': 'header text'}}, {'subheader': {'$in': 'subheaders text'}}, {'context': {'$in': 'context text'}}]}
